# AIML Practical Assignment: Clustering
## Customer Segmentation using K-Means, Agglomerative Clustering, and DBSCAN

**Dataset:** Kaggle Mall Customer Segmentation Dataset (`Mall_Customers.csv`).
It contains customer demographic and spending information. This practical uses **Annual Income (k$)** and **Spending Score (1-100)** to discover customer groups.

**Algorithms:** K-Means, Agglomerative (Hierarchical) Clustering, and DBSCAN.

**Evaluation:** Silhouette Score (higher is better), Davies-Bouldin Index (lower is better), Calinski-Harabasz Score (higher is generally better), cluster visualizations, and cluster profiles.

The notebook downloads the public Kaggle dataset directly. No Kaggle API token is required.

## 1. Import Libraries

In [ ]:
import os
import zipfile
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Download and Load Kaggle Dataset

In [ ]:
!wget -q https://www.kaggle.com/api/v1/datasets/download/vjchoudhary7/customer-segmentation-tutorial-in-python -O mall_customers.zip
!unzip -o -q mall_customers.zip

csv_files = glob.glob("**/Mall_Customers.csv", recursive=True)
if not csv_files:
    csv_files = glob.glob("**/*.csv", recursive=True)
if not csv_files:
    raise FileNotFoundError("No CSV file was found after extracting the Kaggle dataset.")

df = pd.read_csv(csv_files[0])
print("Dataset shape:", df.shape)
display(df.head())
print("\nMissing values:\n", df.isnull().sum())

## 3. Select Features and Explore Dataset

In [ ]:
# Annual income and spending score are used to discover customer spending groups.
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[features].copy()

print("Selected feature shape:", X.shape)
display(X.describe())

plt.figure(figsize=(8, 5))
sns.scatterplot(data=X, x=features[0], y=features[1])
plt.title("Mall Customers: Annual Income vs Spending Score")
plt.tight_layout()
plt.show()

## 4. Standardize the Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Standardized feature means:", np.round(X_scaled.mean(axis=0), 3))
print("Standardized feature standard deviations:", np.round(X_scaled.std(axis=0), 3))

## 5. Select K for K-Means using the Elbow Method

In [ ]:
inertias = []
k_values = range(2, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia / WCSS")
plt.title("Elbow Method for K-Means")
plt.xticks(list(k_values))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Implement K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
df["KMeans_Cluster"] = kmeans_labels

print("K-Means cluster counts:")
print(pd.Series(kmeans_labels).value_counts().sort_index())

## 7. Implement Agglomerative (Hierarchical) Clustering

In [ ]:
agg_model = AgglomerativeClustering(n_clusters=5, linkage="ward")
agg_labels = agg_model.fit_predict(X_scaled)
df["Agglomerative_Cluster"] = agg_labels

print("Agglomerative cluster counts:")
print(pd.Series(agg_labels).value_counts().sort_index())

## 8. Dendrogram for Hierarchical Clustering

In [ ]:
linked = linkage(X_scaled, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(linked, truncate_mode="lastp", p=25, leaf_rotation=45, leaf_font_size=9)
plt.title("Hierarchical Clustering Dendrogram (Truncated)")
plt.xlabel("Merged customer groups")
plt.ylabel("Euclidean distance")
plt.tight_layout()
plt.show()

## 9. Implement DBSCAN Clustering

In [ ]:
# DBSCAN finds dense regions and marks outliers as noise (-1).
# eps and min_samples are chosen for this standardized two-feature dataset.
dbscan = DBSCAN(eps=0.35, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)
df["DBSCAN_Cluster"] = dbscan_labels

print("DBSCAN cluster counts (including -1 noise):")
print(pd.Series(dbscan_labels).value_counts().sort_index())

## 10. Evaluate Clustering Quality

In [ ]:
def clustering_metrics(X_values, labels, model_name):
    labels = np.asarray(labels)
    # Ignore noise points (-1) for DBSCAN's cluster quality metrics.
    valid_mask = labels != -1
    X_eval = X_values[valid_mask]
    labels_eval = labels[valid_mask]
    n_clusters = len(set(labels_eval))

    if n_clusters < 2 or n_clusters >= len(labels_eval):
        return {
            "Model": model_name,
            "Clusters (excluding noise)": n_clusters,
            "Silhouette Score": np.nan,
            "Davies-Bouldin Index": np.nan,
            "Calinski-Harabasz Score": np.nan,
            "Noise Points": int(np.sum(labels == -1))
        }

    return {
        "Model": model_name,
        "Clusters (excluding noise)": n_clusters,
        "Silhouette Score": silhouette_score(X_eval, labels_eval),
        "Davies-Bouldin Index": davies_bouldin_score(X_eval, labels_eval),
        "Calinski-Harabasz Score": calinski_harabasz_score(X_eval, labels_eval),
        "Noise Points": int(np.sum(labels == -1))
    }

metrics = pd.DataFrame([
    clustering_metrics(X_scaled, kmeans_labels, "K-Means"),
    clustering_metrics(X_scaled, agg_labels, "Agglomerative"),
    clustering_metrics(X_scaled, dbscan_labels, "DBSCAN")
])

display(metrics.round(4))

## 11. Visualize Clusters for Each Algorithm

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_data = [
    ("K-Means", kmeans_labels),
    ("Agglomerative", agg_labels),
    ("DBSCAN", dbscan_labels)
]

for ax, (name, labels) in zip(axes, plot_data):
    scatter = ax.scatter(
        X[features[0]], X[features[1]],
        c=labels, cmap="viridis", s=45, alpha=0.85
    )
    ax.set_title(f"{name} Clusters")
    ax.set_xlabel(features[0])
    ax.set_ylabel(features[1])

plt.suptitle("Visual Comparison of Clustering Algorithms", fontsize=14)
plt.tight_layout()
plt.show()

## 12. Cluster Profiles for K-Means

In [ ]:
kmeans_profile = df.groupby("KMeans_Cluster")[features].mean().round(2)
kmeans_profile["Number of Customers"] = df["KMeans_Cluster"].value_counts().sort_index()
display(kmeans_profile)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df, x=features[0], y=features[1],
    hue="KMeans_Cluster", palette="tab10", s=70
)
plt.title("K-Means Customer Segments")
plt.tight_layout()
plt.show()

## 13. Compare Metrics with Graphs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=metrics, x="Model", y="Silhouette Score", ax=axes[0])
axes[0].set_title("Silhouette Score (Higher is Better)")
sns.barplot(data=metrics, x="Model", y="Davies-Bouldin Index", ax=axes[1])
axes[1].set_title("Davies-Bouldin Index (Lower is Better)")
sns.barplot(data=metrics, x="Model", y="Calinski-Harabasz Score", ax=axes[2])
axes[2].set_title("Calinski-Harabasz Score (Higher is Better)")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 14. Findings and Interpretation

In [ ]:
print("Clustering evaluation summary:")
display(metrics.round(4))

print("\nInterpretation guide:")
print("- Higher Silhouette Score indicates better-separated, compact clusters.")
print("- Lower Davies-Bouldin Index indicates better cluster separation.")
print("- Higher Calinski-Harabasz Score generally indicates dense and separated clusters.")
print("- DBSCAN label -1 represents noise/outlier observations.")
print("- Review the scatter plots and cluster profiles before assigning business meanings.")

## 15. Conclusion and Further Improvements

- **K-Means:** suitable when the number of clusters is known or can be estimated, and clusters are relatively compact. It is efficient but sensitive to outliers and the selected value of k.
- **Agglomerative Clustering:** useful for exploring hierarchical relationships and does not require random centroid initialization. It can be computationally expensive on large datasets.
- **DBSCAN:** useful for discovering irregularly shaped dense groups and identifying noise/outliers. Its result depends strongly on `eps` and `min_samples`, and it may struggle when cluster densities vary.

The metrics and plots should be interpreted together. Silhouette score is higher-is-better; Davies-Bouldin is lower-is-better; Calinski-Harabasz is generally higher-is-better. Do not assign a business meaning to a cluster from its numeric label alone; inspect its feature profile.

**Further improvements:** tune `k`, `eps`, and `min_samples`; compare additional features; test other scaling methods; use PCA for high-dimensional visualization; and validate segments on a larger customer dataset.

## Screenshot Checklist

Capture genuine screenshots after running the notebook:
1. Dataset shape, head, and missing values
2. Feature scatter plot
3. Elbow graph
4. K-Means cluster counts
5. Agglomerative cluster counts and dendrogram
6. DBSCAN cluster counts
7. Clustering metrics table
8. Cluster visualization comparison
9. K-Means cluster profile
10. Metric comparison graphs and interpretation